# Mushroom Parser — Ad-hoc исследования
Блокнот для разведочного анализа данных на каждом этапе пайплайна.

In [ ]:
import pandas as pd
import json
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

pd.set_option('display.max_colwidth', 200)
pd.set_option('display.max_rows', 50)

## 1. Сырые посты

In [ ]:
with open('data/raw_posts.json', encoding='utf-8') as f:
    posts = json.load(f)

df_raw = pd.DataFrame(posts)
df_raw['date_posted'] = pd.to_datetime(df_raw['date_posted'])
print(f'Всего постов: {len(df_raw)}')
df_raw.head()

In [ ]:
# Посты по годам
df_raw.groupby(df_raw['date_posted'].dt.year).size().rename('posts').to_frame()

In [ ]:
# Посты без текста
no_text = df_raw[df_raw['text'].str.strip() == '']
print(f'Постов без текста: {len(no_text)} ({len(no_text)/len(df_raw)*100:.1f}%)')

## 2. Анализ распознавания дат

In [ ]:
df = pd.read_csv('data/posts_with_dates.csv', parse_dates=['date_posted', 'foray_date'])
print(f'Всего: {len(df)}')
df['date_source'].value_counts()

In [ ]:
# Посты БЕЗ распознанной даты — смотрим тексты
no_date = df[df['foray_date'].isna() & (df['text_preview'].str.len() > 10)]
print(f'Без даты: {len(no_date)}')
no_date['text_preview'].head(30)

In [ ]:
# Разница между датой поста и датой похода (в днях)
df_valid = df[df['foray_date'].notna()].copy()
df_valid['lag_days'] = (df_valid['date_posted'] - df_valid['foray_date']).dt.days
print(df_valid['lag_days'].describe())
# Подозрительные — слишком большой разрыв
df_valid[df_valid['lag_days'] > 30][['date_posted', 'foray_date', 'lag_days', 'text_preview']].head(10)

## 3. Агрегированные отчёты по датам

In [ ]:
daily = pd.read_csv('data/daily_counts.csv', parse_dates=['date'])
daily = daily[daily['date'] >= '2020-01-01']
print(f'Уникальных дат: {len(daily)}')
daily.describe()

In [ ]:
# Топ-20 дней по числу отчётов
daily.nlargest(20, 'report_count')[['date', 'report_count', 'year']]

In [ ]:
# Сезонность: среднее число отчётов по месяцам
daily.groupby('month')['report_count'].mean().round(2)

## 4. Графики активности

In [ ]:
daily = pd.read_csv('data/daily_counts.csv', parse_dates=['date'])
daily = daily[daily['date'] >= '2020-01-01']

# ── По дням ────────────────────────────────────────────────────────────────────
fig = go.Figure()
fig.add_trace(go.Bar(
    x=daily['date'], y=daily['report_count'],
    marker_color='#2196F3', name='отчётов'
))
fig.update_layout(
    title='Количество отчётов по дням',
    xaxis_title='Дата', yaxis_title='Отчётов',
    hovermode='x unified', height=400,
    plot_bgcolor='white',
    yaxis=dict(gridcolor='#e0e0e0'),
    xaxis=dict(gridcolor='#e0e0e0'),
)
fig.show()

In [ ]:
# ── По неделям ─────────────────────────────────────────────────────────────────
weekly = daily.set_index('date')['report_count'].resample('W').sum().reset_index()

fig = go.Figure()
fig.add_trace(go.Bar(
    x=weekly['date'], y=weekly['report_count'],
    marker_color='#FF6F00', name='отчётов'
))
fig.update_layout(
    title='Количество отчётов по неделям',
    xaxis_title='Дата', yaxis_title='Отчётов',
    hovermode='x unified', height=400,
    plot_bgcolor='white',
    yaxis=dict(gridcolor='#e0e0e0'),
    xaxis=dict(gridcolor='#e0e0e0'),
)
fig.show()

In [ ]:
# ── Сезонность: каждый год отдельной линией (грибной сезон, недели 14–48) ──────
COLORS = ['#e6194b','#3cb44b','#4363d8','#f58231','#911eb4','#42d4f4','#f032e6']

fig = go.Figure()
for i, year in enumerate(sorted(daily['year'].unique())):
    year_data = daily[daily['year'] == year].copy()
    weekly_yr = year_data.set_index('date')['report_count'].resample('W').sum().reset_index()
    weekly_yr['week'] = weekly_yr['date'].dt.isocalendar().week.astype(int)
    # Только грибной сезон
    weekly_yr = weekly_yr[weekly_yr['week'].between(14, 48)]

    fig.add_trace(go.Scatter(
        x=weekly_yr['week'], y=weekly_yr['report_count'],
        mode='lines', name=str(year),
        line=dict(width=2.5, color=COLORS[i % len(COLORS)]),
    ))

fig.update_layout(
    title='Сезонность — каждый год отдельно (апрель–ноябрь)',
    xaxis=dict(title='Неделя года', range=[14, 48], dtick=2, gridcolor='#e0e0e0'),
    yaxis=dict(title='Отчётов', gridcolor='#e0e0e0'),
    hovermode='x unified', height=450,
    plot_bgcolor='white',
    legend=dict(orientation='v', x=1.01, y=1),
)
fig.show()

## 4. Погода

In [ ]:
# Корреляция погодных фич с количеством отчётов
num_cols = season.select_dtypes('number').columns.drop(['year', 'month', 'day_of_year', 'weekday'])
corr = season[num_cols].corr()['report_count'].drop('report_count').sort_values(key=abs, ascending=False)

# График
top = corr.head(25)
colors = ['#2196F3' if v >= 0 else '#e6194b' for v in top.values]

fig = go.Figure(go.Bar(
    x=top.values,
    y=top.index,
    orientation='h',
    marker_color=colors,
))
fig.update_layout(
    title='Топ-25 фич по корреляции Пирсона с количеством отчётов',
    xaxis_title='Корреляция',
    yaxis=dict(autorange='reversed'),
    height=650,
    plot_bgcolor='white',
    xaxis=dict(gridcolor='#e0e0e0', zeroline=True, zerolinecolor='black', zerolinewidth=1),
)
fig.show()

In [ ]:
# Scatter: топ-фича vs количество отчётов
top_feature = corr.index[0]

fig = px.scatter(
    season, x=top_feature, y='report_count',
    color='year', opacity=0.5,
    trendline='ols',
    title=f'Зависимость отчётов от «{top_feature}»',
    labels={top_feature: top_feature, 'report_count': 'Отчётов', 'year': 'Год'},
    color_continuous_scale='Viridis',
)
fig.update_layout(height=450, plot_bgcolor='white')
fig.show()

In [ ]:
weather.describe()

## 5. Объединённый датасет (грибы + погода)

In [ ]:
# Джойним отчёты с погодой
merged = weather.merge(daily[['date', 'report_count']], on='date', how='left')
merged['report_count'] = merged['report_count'].fillna(0).astype(int)

# Оставляем только грибной сезон (апрель–ноябрь)
season = merged[merged['month'].between(4, 11)]
print(f'Дней в сезоне: {len(season)}, из них с отчётами: {(season["report_count"] > 0).sum()}')
season.head()

In [ ]:
# Корреляция погодных фич с количеством отчётов
num_cols = season.select_dtypes('number').columns.drop(['year', 'month', 'day_of_year', 'weekday'])
corr = season[num_cols].corr()['report_count'].drop('report_count').sort_values(key=abs, ascending=False)
corr.head(20)

## 6. Модель LightGBM

In [ ]:
import importlib
import train_model
importlib.reload(train_model)
from train_model import train, TEST_YEAR
import numpy as np

model, importance, preds_df, train_df_preds = train()

In [ ]:
# ── Предсказания vs реальность (трейн + тест) ─────────────────────────────────
fig = go.Figure()

# Трейн — факт
fig.add_trace(go.Scatter(
    x=train_df_preds['date'], y=train_df_preds['report_count'],
    mode='lines', name='Факт (трейн)',
    line=dict(color='#90CAF9', width=1.5),
    opacity=0.8,
))
# Трейн — прогноз
fig.add_trace(go.Scatter(
    x=train_df_preds['date'], y=train_df_preds['predicted'],
    mode='lines', name='Прогноз (трейн)',
    line=dict(color='#EF9A9A', width=1.5, dash='dot'),
    opacity=0.8,
))

# Разделитель трейн/тест
fig.add_vline(
    x=preds_df['date'].min().timestamp() * 1000,
    line_dash='dash', line_color='#555', line_width=1.5,
    annotation_text=f'Тест ({TEST_YEAR})', annotation_position='top right',
)

# Тест — факт
fig.add_trace(go.Scatter(
    x=preds_df['date'], y=preds_df['report_count'],
    mode='lines', name='Факт (тест)',
    line=dict(color='#2196F3', width=2),
))
# Тест — прогноз
fig.add_trace(go.Scatter(
    x=preds_df['date'], y=preds_df['predicted'],
    mode='lines', name='Прогноз (тест)',
    line=dict(color='#e6194b', width=2, dash='dot'),
))

fig.update_layout(
    title='Факт vs Прогноз — трейн и тест',
    xaxis_title='Дата', yaxis_title='Отчётов',
    hovermode='x unified', height=450,
    plot_bgcolor='white',
    yaxis=dict(gridcolor='#e0e0e0'),
    xaxis=dict(gridcolor='#e0e0e0'),
    legend=dict(orientation='h', y=-0.2),
)
fig.show()

In [ ]:
# ── Feature importance — топ-30 ───────────────────────────────────────────────
top_imp = importance.head(30)
colors = ['#2196F3'] * len(top_imp)

fig = go.Figure(go.Bar(
    x=top_imp.values,
    y=top_imp.index,
    orientation='h',
    marker_color=colors,
))
fig.update_layout(
    title='Feature Importance LightGBM (топ-30)',
    xaxis_title='Importance (split)',
    yaxis=dict(autorange='reversed'),
    height=700,
    plot_bgcolor='white',
    xaxis=dict(gridcolor='#e0e0e0'),
)
fig.show()

In [ ]:
# ── Scatter: факт vs прогноз (насколько хорошо калиброван?) ──────────────────
from sklearn.metrics import r2_score
r2 = r2_score(preds_df['report_count'], preds_df['predicted'])

fig = px.scatter(
    preds_df, x='report_count', y='predicted',
    opacity=0.6,
    labels={'report_count': 'Факт', 'predicted': 'Прогноз'},
    title=f'Калибровка модели — R² = {r2:.3f}',
    trendline='ols',
    color_discrete_sequence=['#4363d8'],
)
# Линия идеального прогноза
max_val = max(preds_df['report_count'].max(), preds_df['predicted'].max()) + 5
fig.add_trace(go.Scatter(
    x=[0, max_val], y=[0, max_val],
    mode='lines', name='Идеал',
    line=dict(color='#e6194b', dash='dash', width=1.5),
))
fig.update_layout(height=450, plot_bgcolor='white')
fig.show()